# GMRES Convergence Controls

![Highly scattering vacuum slab used for the GMRES study](images/gmres_scattering_slab.png)

**Goal:** Show how the inner tolerance and restart interval affect GMRES accuracy, transport sweeps, and wall time.

## Scattering slab

A finite slab with vacuum boundaries and a scattering ratio of 0.99 gives GMRES enough spatial error modes for its controls to matter. Every case uses the same 1000-cell mesh, 32-direction quadrature, and source. A tightly converged solve supplies the reference flux.

In [ ]:
from mpi4py import MPI
from pyopensn.aquad import GLProductQuadrature1DSlab
from pyopensn.context import Finalize
from pyopensn.mesh import OrthogonalMeshGenerator
from pyopensn.post import VolumePostprocessor
from pyopensn.solver import DiscreteOrdinatesProblem, SteadyStateSourceSolver
from pyopensn.source import VolumetricSource
from pyopensn.xs import MultiGroupXS

comm = MPI.COMM_WORLD
rank = comm.rank


def solve(tolerance, restart):
    mesh = OrthogonalMeshGenerator(
        node_sets=[[10.0 * i / 1000 for i in range(1001)]]
    ).Execute()
    mesh.SetUniformBlockID(0)
    xs = MultiGroupXS()
    xs.CreateSimpleOneGroup(sigma_t=1.0, c=0.99)
    quadrature = GLProductQuadrature1DSlab(
        n_polar=32, scattering_order=0
    )
    problem = DiscreteOrdinatesProblem(
        mesh=mesh,
        num_groups=1,
        groupsets=[
            {
                "groups_from_to": (0, 0),
                "angular_quadrature": quadrature,
                "inner_linear_method": "petsc_gmres",
                "l_abs_tol": tolerance,
                "l_max_its": 500,
                "gmres_restart_interval": restart,
            }
        ],
        xs_map=[{"block_ids": [0], "xs": xs}],
        volumetric_sources=[
            VolumetricSource(block_ids=[0], group_strength=[1.0])
        ],
        boundary_conditions=[
            {"name": "zmin", "type": "vacuum"},
            {"name": "zmax", "type": "vacuum"},
        ],
        options={"verbose_inner_iterations": False},
    )

    solver = SteadyStateSourceSolver(problem=problem)
    comm.Barrier()
    start = MPI.Wtime()
    solver.Initialize()
    solver.Execute()
    comm.Barrier()
    elapsed = comm.allreduce(MPI.Wtime() - start, op=MPI.MAX)

    average = VolumePostprocessor(problem=problem, value_type="avg")
    average.Execute()
    flux = sum(average.GetValue()[0])
    return flux, solver.GetNumSweeps(), elapsed

## Vary tolerance and restart

The tolerance cases use a restart interval of 30. The restart cases use a tolerance of $10^{-8}$. Sweep counts measure transport work; wall time is machine-dependent.

In [ ]:
reference_flux, reference_sweeps, reference_time = solve(1.0e-10, 50)
cases = [
    ("Tolerance 1e-4", 1.0e-4, 30),
    ("Tolerance 1e-6", 1.0e-6, 30),
    ("Tolerance 1e-8", 1.0e-8, 30),
    ("Restart 5", 1.0e-8, 5),
    ("Restart 20", 1.0e-8, 20),
    ("Restart 50", 1.0e-8, 50),
]
results = {label: solve(tolerance, restart) for label, tolerance, restart in cases}
differences = {
    label: abs(flux - reference_flux) / abs(reference_flux)
    for label, (flux, _, _) in results.items()
}

if rank == 0:
    print(f"GMRES reference total average flux={reference_flux:.12e}")
    print(f"GMRES reference sweeps={reference_sweeps}")
    print(f"GMRES reference wall time (s)={reference_time:.6f}")
    for label, _, _ in cases:
        flux, sweeps, elapsed = results[label]
        print(f"GMRES {label} relative flux difference={differences[label]:.12e}")
        print(f"GMRES {label} sweeps={sweeps}")
        print(f"GMRES {label} wall time (s)={elapsed:.6f}")

assert max(differences.values()) < 1.0e-3
assert results["Tolerance 1e-4"][1] <= results["Tolerance 1e-8"][1]

A representative one-process run gives:

| Case | Relative flux difference | Sweeps | Wall time (s) |
|---|---:|---:|---:|
| Reference ($10^{-10}$, restart 50) | 0 | 17 | 0.0183 |
| Tolerance $10^{-4}$ | $3.74\times10^{-9}$ | 11 | 0.0119 |
| Tolerance $10^{-6}$ | $1.40\times10^{-11}$ | 13 | 0.0136 |
| Tolerance $10^{-8}$ | $1.34\times10^{-13}$ | 15 | 0.0155 |
| Restart 5 | $1.52\times10^{-9}$ | 41 | 0.0418 |
| Restart 20 | $1.34\times10^{-13}$ | 15 | 0.0157 |
| Restart 50 | $1.34\times10^{-13}$ | 15 | 0.0158 |

Tighter tolerances require more sweeps. A restart interval of 5 discards useful Krylov information and nearly triples the work relative to intervals of 20 or 50. Exact timings depend on the machine.

## Finalize (for Jupyter Notebook only)

In script mode, PyOpenSn handles finalization automatically. In a Jupyter kernel, finalize OpenSn before MPI.

In [ ]:
if "opensn_console" not in globals():
    from IPython import get_ipython

    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()